<a href="https://colab.research.google.com/github/muhammedriswanp/ai-ml-learning-lab/blob/main/WEEK6_AI_Agent_Orchestration_Frameworks/CrewAI_Role_Based_Multi_Agent_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market Entry Research Assistant

Develop a Market Entry Research Assistant with three collaborating agents:

## Agents

### 1. Research Agent
*   Collect market information
*   Gather competitor data
*   Identify customer segments

### 2. Business Analyst
*   Analyse collected information
*   Perform SWOT analysis
*   Estimate opportunities and risks

### 3. Report Writer
*   Generate an executive summary
*   Recommend whether expansion should proceed

### Example Scenario:
"Should a Direct-to-Consumer clothing brand expand into Tier-2 cities in India?"

## Suggested Dataset
*   Business problem statement
*   Public reports
*   Government statistics
*   Online market research

## Deliverables
*   CrewAI implementation
*   Final executive summary
*   Agent interaction logs
*   High-level architecture diagram

In [1]:
!pip install langchain-groq crewai litellm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [22]:
import os
from langchain_groq import ChatGroq
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM


os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

groq_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.7,
    cache_prompt=False  # <--- Explicitly turns off cache_breakpoint parameters globally
)



### Research Agent

In [24]:
research_agent = Agent(
    role="Market Research Specialist",
    goal="Collect comprehensive market information about Tier-2 cities in India for a Direct-to-Consumer clothing brand",
    backstory="""You are an experienced market research specialist with 10 years of
    experience analysing Indian consumer markets. You have deep knowledge of Tier-2
    city demographics, spending patterns, and retail trends. You are thorough,
    data-driven, and always back your findings with specific insights.""",
    llm=groq_llm,
    cache_prompt=False,
    verbose=True,
    allow_delegation=False
)

print("Research Agent ready")

Research Agent ready


In [25]:
business_analyst = Agent(
    role="Business Strategy Analyst",
    goal="Analyse market research findings and perform a detailed SWOT analysis with opportunity and risk assessment for Tier-2 city expansion",
    backstory="""You are a sharp business strategy analyst with an MBA and 8 years
    of experience in retail and e-commerce consulting across South Asia. You specialise
    in market entry strategies, competitive analysis, and risk assessment. You think
    critically, challenge assumptions, and always quantify opportunities where possible.
    You rely on the research provided to you and build structured, actionable insights.""",
    llm=groq_llm,
    cache_prompt=False,
    verbose=True,
    allow_delegation=False
)

print("Business Analyst ready")

Business Analyst ready


In [26]:
report_writer = Agent(
    role="Business Report Writer",
    goal="Transform research and analysis into a clear, compelling executive summary with a final recommendation on Tier-2 city expansion",
    backstory="""You are a senior business report writer with 12 years of experience
    crafting executive summaries for C-suite decision makers. You have worked with
    top consulting firms and know how to distill complex analysis into clear, concise,
    and actionable reports. Your writing is professional, structured, and always ends
    with a definitive recommendation backed by evidence.""",
    llm=groq_llm,
    cache_prompt=False,
    verbose=True,
    allow_delegation=False
)

print("Report Writer ready")

Report Writer ready


# Define Tasks

In [8]:
research_task = Task(
    description="""Research the market opportunity for a Direct-to-Consumer (DTC)
    clothing brand expanding into Tier-2 cities in India. Cover the following:

    1. Market size and growth trends of Tier-2 cities
    2. Demographics and spending behaviour of Tier-2 consumers
    3. Top 3 competitors already operating in this space
    4. Key customer segments (age, income, lifestyle)
    5. Popular clothing categories and price sensitivity

    Be specific with data points and insights wherever possible.""",
    expected_output="""A structured market research report with 5 sections:
    market overview, consumer demographics, competitor landscape,
    customer segments, and pricing insights.""",
    agent=research_agent
)

print("Research Task ready")

Research Task ready


In [9]:
analysis_task = Task(
    description="""Using the market research provided, perform a comprehensive
    business analysis for the DTC clothing brand's Tier-2 city expansion. Cover:

    1. SWOT Analysis (Strengths, Weaknesses, Opportunities, Threats)
    2. Top 3 opportunities with estimated market potential
    3. Top 3 risks with likelihood and impact rating (High/Medium/Low)
    4. Competitive positioning strategy
    5. Key success factors for Tier-2 market entry

    Base your analysis strictly on the research findings provided to you.""",
    expected_output="""A structured business analysis with:
    - SWOT table
    - Opportunity assessment
    - Risk register
    - Positioning strategy
    - Success factors""",
    agent=business_analyst,
    context=[research_task]
)

print("Analysis Task ready")

Analysis Task ready


In [10]:
report_task = Task(
    description="""Using the market research and business analysis provided,
    write a professional executive summary report for the leadership team of
    a DTC clothing brand considering Tier-2 city expansion in India. Include:

    1. Executive Summary (2-3 paragraphs)
    2. Market Opportunity Overview
    3. Key Findings from Research
    4. SWOT Highlights
    5. Risks and Mitigation Strategies
    6. Final Recommendation — Should the brand expand? Yes or No, and why.

    Write in a professional tone suitable for C-suite executives.""",
    expected_output="""A complete executive report with 6 sections, ending with
    a clear Yes/No expansion recommendation backed by evidence from
    the research and analysis.""",
    agent=report_writer,
    context=[research_task, analysis_task]
)

print("Report Task ready")

Report Task ready


# Assemble & Run the Crew

In [29]:
import crewai.llms.cache as _crewai_cache

# Overwrite the cache_breakpoint injector with a clean no-op function
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

print("Successfully patched CrewAI cache injection engine for Groq!")


Successfully patched CrewAI cache injection engine for Groq!


In [30]:
crew = Crew(
    agents=[research_agent, business_analyst, report_writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.sequential,
    verbose=True
)

print("Crew assembled. Starting execution...\n")
result = await crew.kickoff_async()

print("\n" + "="*60)
print("FINAL REPORT")
print("="*60)
print(result)

Crew assembled. Starting execution...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fd76f37d-708a-4aa3-8492-4ec00fce5f08                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the market opportunity for a Direct-to-Consumer (DTC)                                           │
│      clothing brand expanding into Tier-2 cities in India. Cover the following:                                 │
│                                                                                                                 │
│      1. Market size and growth trends of Tier-2 cities                                                          │
│      2. Demographics and spending behaviour of Tier-2 consumers                                                 │
│      3. Top 3 competitors already operating in this space                                                       │
│      4. Key customer segments (age, income, lifestyle)                                                          │
│      5. Popular clothing categories and price sensitivity                                                       │
│                                                                                                                 │
│      Be specific with data points and insights wherever possible.                                               │
│  ID: 2dd84f82-6faf-43be-9c7e-3710d40d41d6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Specialist                                                                              │
│                                                                                                                 │
│  Task: Research the market opportunity for a Direct-to-Consumer (DTC)                                           │
│      clothing brand expanding into Tier-2 cities in India. Cover the following:                                 │
│                                                                                                                 │
│      1. Market size and growth trends of Tier-2 cities                                                          │
│      2. Demographics and spending behaviour of Tier-2 consumers                                                 │
│      3. Top 3 competitors already operating in this space                                                       │
│      4. Key customer segments (age, income, lifestyle)                                                          │
│      5. Popular clothing categories and price sensitivity                                                       │
│                                                                                                                 │
│      Be specific with data points and insights wherever possible.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Report Writer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Executive Summary**                                                                                          │
│                                                                                                                 │
│  The Indian Tier-2 cities market presents a significant opportunity for a Direct-to-Consumer (DTC) clothing     │
│  brand, driven by growing internet penetration, increasing smartphone adoption, and rising disposable incomes.  │
│  Our market research and business analysis indicate that the Tier-2 cities market is expected to grow at a      │
│  CAGR of 12% from 2023 to 2028, with an estimated market size of ₹5,000 crores (approximately $667 million      │
│  USD) by 2028. The key drivers of this growth include a growing middle class, increasing online shopping, and   │
│  a preference for quality and fashionable clothing.                                                             │
│                                                                                                                 │
│  The DTC clothing brand can capitalize on this opportunity by positioning itself as a quality and value-driven  │
│  brand, offering personalized and customized products, and leveraging digital platforms to engage with          │
│  customers and build brand awareness. However, the brand must also navigate the competitive landscape,          │
│  characterized by established players like Bewakoof, FabAlley, and Roadster, and mitigate risks such as high    │
│  price sensitivity, fluctuations in raw material costs, and regulatory changes.                                 │
│                                                                                                                 │
│  Our analysis suggests that the DTC clothing brand can achieve success in the Tier-2 cities market by           │
│  understanding local consumer preferences, offering competitive pricing and quality, and establishing a strong  │
│  supply chain and logistics network. With a well-planned market entry strategy, the brand can tap into the      │
│  growing demand for online shopping, fashionable clothing, and sustainable products, ultimately achieving       │
│  significant revenue growth and market share.                                                                   │
│                                                                                                                 │
│  The Tier-2 cities market in India offers a unique opportunity for the DTC clothing brand to expand its         │
│  customer base, increase revenue, and establish a strong presence in the Indian market. With a thorough         │
│  understanding of the market dynamics, consumer preferences, and competitive landscape, the brand can develop   │
│  a targeted strategy to capitalize on this opportunity and achieve long-term success.                           │
│                                                                                                                 │
│  **Market Opportunity Overview**                                                                                │
│                                                                                                                 │
│  The Tier-2 cities market in India offers a significant opportunity for the DTC clothing brand, driven by a     │
│  growing middle class, increasing internet penetration,

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the market research and business analysis provided,                                                │
│      write a professional executive summary report for the leadership team of                                   │
│      a DTC clothing brand considering Tier-2 city expansion in India. Include:                                  │
│                                                                                                                 │
│      1. Executive Summary (2-3 paragraphs)                                                                      │
│      2. Market Opportunity Overview                                                                             │
│      3. Key Findings from Research                                                                              │
│      4. SWOT Highlights                                                                                         │
│      5. Risks and Mitigation Strategies                                                                         │
│      6. Final Recommendation — Should the brand expand? Yes or No, and why.                                     │
│                                                                                                                 │
│      Write in a professional tone suitable for C-suite executives.                                              │
│  Agent: Business Report Writer                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL REPORT
**Executive Summary**

The Indian Tier-2 cities market presents a significant opportunity for a Direct-to-Consumer (DTC) clothing brand, driven by growing internet penetration, increasing smartphone adoption, and rising disposable incomes. Our market research and business analysis indicate that the Tier-2 cities market is expected to grow at a CAGR of 12% from 2023 to 2028, with an estimated market size of ₹5,000 crores (approximately $667 million USD) by 2028. The key drivers of this growth include a growing middle class, increasing online shopping, and a preference for quality and fashionable clothing.

The DTC clothing brand can capitalize on this opportunity by positioning itself as a quality and value-driven brand, offering personalized and customized products, and leveraging digital platforms to engage with customers and build brand awareness. However, the brand must also navigate the competitive landscape, characterized by established players like Bewakoof, FabAlle

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯